In [ ]:
!pip install --no-cache-dir "pillow==10.3.0" "torch>=2.0.0" "transformers>=4.40.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 56.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [11]:
import os
import torch
import pandas as pd
from PIL import Image
from IPython.display import display
from sentence_transformers import SentenceTransformer, util

from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

In [8]:
data_root = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset"

styles_path = os.path.join(data_root, "styles.csv")
image_dir   = os.path.join(data_root, "images")

df = pd.read_csv(styles_path, on_bad_lines="skip")

print(df.columns)  # to confirm id, productDisplayName, etc.
df.head()

Index(['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
       'baseColour', 'season', 'year', 'usage', 'productDisplayName'],
      dtype='object')


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [12]:
def image_path_from_id(pid):
    # Many fashion datasets name images as "<id>.jpg"
    path_jpg = os.path.join(image_dir, f"{pid}.jpg")
    path_png = os.path.join(image_dir, f"{pid}.png")

    if os.path.exists(path_jpg):
        return path_jpg
    if os.path.exists(path_png):
        return path_png
    return None

df["image_path"] = df["id"].apply(image_path_from_id)

# Keep only rows where the image exists
df = df[df["image_path"].notna()].reset_index(drop=True)

df[["id", "productDisplayName", "image_path"]].head()

,id,productDisplayName,image_path
0,15970,Turtle Check Men Navy Blue Shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,39386,Peter England Men Party Blue Jeans,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,59263,Titan Women Silver Watch,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,21379,Manchester United Men Solid Black Track Pants,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,53759,Puma Men Grey T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...


In [13]:
model_name = "sentence-transformers/clip-ViT-B-32"
model = SentenceTransformer(model_name)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

0_CLIPModel/model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (3 outputs left out)

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [14]:
N = 2000  # or smaller/larger depending on speed
df_subset = df.head(N).copy()

images = [Image.open(p).convert("RGB") for p in df_subset["image_path"].tolist()]

image_embeddings = model.encode(
    images,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True
)

image_embeddings.shape

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

torch.Size([2000, 512])

In [15]:
def search_images_by_text(query, top_k=5):
    # 1) Encode text query
    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    # 2) Cosine similarity with image embeddings
    cos_scores = util.cos_sim(query_embedding, image_embeddings)[0]

    # 3) Top-k indices
    top_results = torch.topk(cos_scores, k=top_k)

    results = []
    for score, idx in zip(top_results.values, top_results.indices):
        idx = idx.item()
        row = df_subset.iloc[idx]
        results.append({
            "row_index": idx,
            "id": row["id"],
            "name": row.get("productDisplayName", ""),
            "image_path": row["image_path"],
            "score": float(score)
        })
    return results

In [29]:
import math
import matplotlib.pyplot as plt

def show_search_results_grid(results, thumb_size=(128, 128), rows=2, cols=5):
    n = len(results)
    if n == 0:
        print("No results.")
        return

    # If more results than slots, cut to rows*cols
    max_slots = rows * cols
    results = results[:max_slots]
    n = len(results)

    plt.figure(figsize=(cols * 2, rows * 2))

    for i, r in enumerate(results):
        img = Image.open(r["image_path"]).convert("RGB")
        img = img.resize(thumb_size)

        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img)
        ax.set_title(str(r["id"]), fontsize=7)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [31]:
while True:
    q = input("Enter a text query (or 'exit'): ").strip()
    if q.lower() == "exit":
        break

    results = search_images_by_text(q, top_k=10)
    print("\nTop Matching Images:")
    show_search_results_grid(results, thumb_size=(128, 128), rows=2, cols=5)
    print("-" * 60)

Enter a text query (or 'exit'):  exit
